# 02 — PUBOと小型理想QAOA

Candidate C PUBOと正式な最終標本判定は新しいQuantum APIを使います。qualification decoderは別の独立検査です。回路は教材用のdense対角表現で、正式QPY・COBYLA100評価・2048/4096shots・監督制御の再実行ではありません。

実行方法：notebooks/README.mdに従いRun All。角度・seed・shotsを編集できます。元の科学結果やfreezeを変更しません。必要な配送を省略した電力量減を省エネ成果とせず、量子実機・量子優位性・全需要充足・全システムエネルギーを実証しません。


In [ ]:
from pathlib import Path
import sys, json, math
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/adapters.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリrootまたはnotebooksを作業ディレクトリにしてください')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'notebooks'))
import adapters as api
S = '01_research_design/scenario_evaluation/'
M = '01_research_design/methodology/'
Q = '05_src/traffic_simulation/'
L = '05_src/literature_analysis/'


## 二値変数と古典前処理
顧客×訪問位置の9変数、one-hot制約、旅行時間目的、違反完全順列への高次罰則です。全6順列の時間窓・帰還可行性を正式scheduleで事前評価します。QAOAが枝刈りするわけではありません。

In [ ]:
core = api.candidate_functions()
builder = api.load_module('src/kmd_quantitative/quantum/qualification_builder.py','nb_builder')
decoder = api.load_module('src/kmd_quantitative/quantum/decoder.py','nb_decoder')
protocol = api.load_module(Q+'matched_quantum_r1_c_protocol_v2/protocol.py','nb_protocol')
validator = api.load_module('src/kmd_quantitative/routing/independent.py','nb_ind')
from itertools import permutations
a=api.instance(); records=[]
for order in permutations(range(1,4)):
    tw,ret,starts=builder.schedule(a,order)
    records.append(dict(order=list(order),TIME_WINDOW=tw,RETURN_HORIZON=ret))
model={'n':3,'penalty_s':'1000','capacity_ok':int(a['demand'].sum())<=216,'permutations':records}
q,poly=core['build_C'](model,a)
assert q==9
print('qubits:',q,'PUBO terms:',len(poly),'前評価順列:',len(records))
display(pd.DataFrame(records))
# 完全順列1件だけを違反へ変更し、正式PUBOの罰則差を検査
bad_model={**model, 'permutations':[dict(r,TIME_WINDOW=False) if i==0 else r for i,r in enumerate(records)]}
_,bad_poly=core['build_C'](bad_model,a)
bitcode=core['route_code'](records[0]['order'],3)
def at(poly,code):
    return sum(float(c) for support,c in poly.items() if all(code>>i&1 for i in support))
assert at(bad_poly,bitcode)-at(poly,bitcode)==1000


## コストHamiltonianとp=1回路（軽量教材）
PUBOの各bitstring値を対角要素に持つH_Cから exp(-iγH_C) を構成し、H初期状態とX mixerを適用します。dense表現は512状態限定で、正式回路のゲート資源・拡張性を表しません。
正式QPYは罰則Aで正規化したコスト演算子を使用します。この教材は非正規化dense表現で、同じ角度でも正式回路と同一ではありません。科学的再現ではなくAPI利用例です。


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import DiagonalGate
from qiskit.quantum_info import Statevector
energies=np.zeros(1<<q)
for bits in range(1<<q):
    energies[bits]=sum(float(c) for support,c in poly.items() if all(bits>>i&1 for i in support))
for rec in records:
    route=[0]+rec['order']+[0]
    assert energies[core['route_code'](rec['order'],3)]==sum(a['travel'][i,j] for i,j in zip(route,route[1:]))
def state(theta):
    qc=QuantumCircuit(q);qc.h(range(q))
    qc.append(DiagonalGate(np.exp(-1j*theta[0]*energies)),range(q));qc.rx(2*theta[1],range(q))
    return Statevector.from_instruction(qc)
gamma,beta=0.001,0.2  # 編集可能
shots,seed=64,20261009
assert 1<=shots<=4096
DO_OPTIMIZE=False  # 初期状態では古典最適化しない
if DO_OPTIMIZE:
    from scipy.optimize import minimize
    opt=minimize(lambda x:float(np.dot(state(x).probabilities(),energies)),[gamma,beta],method='COBYLA',bounds=[(0,2*np.pi),(0,np.pi)],options={'maxiter':6})
    gamma,beta=map(float,opt.x)
    print('教材用最大6評価。正式Protocolと異なる:',opt.message)
sv=state([gamma,beta]);assert np.isclose(np.linalg.norm(sv.data),1)
# parameter評価標本と最終取得標本を分離する（各64、独立seed）
sv.seed(seed);probe={int(k,2):int(v) for k,v in sv.sample_counts(shots).items()}
probe_feasible=sum(count for bits,count in probe.items() if decoder.decode(a,[(bits>>i)&1 for i in range(q)],[])['valid'])
evaluation={'evaluation_index':0,'gamma':gamma,'beta':beta,'mean_energy_s':sum(energies[bits]*count for bits,count in probe.items())/shots,'feasible_count':probe_feasible,'optimal_count':0}
selected,no_feasible=protocol.select_incumbent([evaluation])
final_state=state([selected['gamma'],selected['beta']]);final_state.seed(seed+1)
counts={int(k,2):int(v) for k,v in final_state.sample_counts(shots).items()}
assert sum(counts.values())==shots
print('parameter評価標本:',sum(probe.values()),'final標本:',sum(counts.values()))

## 正式な最終標本判定とqualificationとの違い
正式APIは可行経路表へ整数コードを照合し、routing validatorで再検査して、旅行時間→整数ビットコードの順で最小の経路を選びます。標本頻度は最終同点選択の基準ではありません。資格検査用decoderの結果と照合しますが、正式V2がこのdecoderを直接呼ぶわけではありません。可行標本未取得は正常に表示します。


In [ ]:
formal = api.formal_quantum_api()
parsed = formal.parse_sample_counts({format(code,f'0{q}b'):count for code,count in counts.items()}, shots)
final_result = formal.evaluate_final_counts(model, a, parsed)
valid_shots = final_result['final_feasible_count']
best = final_result['best_final_feasible_route']
# qualificationは比較用独立検査。正式判定の代用ではない。
assert valid_shots == sum(count for bits,count in counts.items() if decoder.decode(a,[(bits>>i)&1 for i in range(q)],[])['valid'])
print('可行標本:',valid_shots,'/',shots)
print('最終経路:',best if best else 'NOT_ACQUIRED（取得保証なし）')
print('parameter評価選択 index:',selected['evaluation_index'],'no_feasible:',no_feasible)
assert formal.select_final_route([]) is None


## 限界
角度、seed、shotsを変更できます。初期条件で可行経路がなくても異常終了ではありません。教材回路の成功率を凍結R0/R1結果に混ぜないでください。量子実機・高速化・量子優位性は評価しません。DO_OPTIMIZEの任意分岐は通常の自動試験では実行せずUNVERIFIEDです。